In [10]:
# =============================================================================
# Data Processing
# =============================================================================
import pandas as pd

# =============================================================================
# Text Processing
# =============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
# =============================================================================
# Load the preprocessed dataset
# =============================================================================
dataset = pd.read_csv("preprocessed_dataset.csv")
print(dataset.shape)

dataset.head()

(100000, 13)


,QuestionID,QuestionType,Category,AskerID,QuestionTime,QuestionText,AnswerText,AnswererID,AnswerTime,question_length,answer_length,semantic_text,lexical_text
0,C15Q2112,open-ended,Tools and Home Improvement,A7XRWKF8NWUIP,2013-04-20,- What are the dimensions of this item?,http://www.lockcorporationofamerica.com/produc...,A1I75BSMW757I3,2014-08-19,6.0,31.0,Question: - what are the dimensions of this it...,dimensions item
1,C9Q4595,open-ended,Home and Kitchen,A3RL90G39VP7DD,2014-02-06,How much booze can it hold?,we poured booze into measuring cup then into s...,A3O2W1J9FKGUZC,2014-07-17,6.0,60.0,Question: how much booze can it hold?\n\nAnswe...,much booze hold poured booze measuring cup sun...
2,C4Q7999,open-ended,Cell Phones and Accessories,A3OR47C9CM6ZKW,2014-08-09,Will this case fit Nokia Lumia 520,"yes it fits great, and its a great cover very ...",AOXZ8V0S2WNNC,2014-08-09,6.0,17.0,Question: will this case fit nokia lumia 520\n...,case fit nokia lumia 520 yes fits great great ...
3,C8Q8916,open-ended,Health and Personal Care,A2U5QV3HI50XWC,2014-04-25,"When folded in the sitting position, how high ...",it is 30 inches above the ground....and i stil...,AADRNEZQ5OWBK,2014-04-25,6.0,7.0,"Question: when folded in the sitting position,...",folded sitting position high ground seat 30 in...
4,C14Q905,open-ended,Sports and Outdoors,A1H0DB767YZGUJ,2015-04-15,How long should I leave this on to get the max...,"I washed my sauba shirt in the ""delicate"" cycl...",A53Y6WZ8KBN49,2013-12-23,6.0,24.0,Question: how long should i leave this on to g...,long leave get max sweat benefit keep thinking...


In [12]:
# ──────────────────────────────────────────────────────────────────────────────
# Semantic Chunking
# Step 1: Define a sliding-window chunking function
# ──────────────────────────────────────────────────────────────────────────────
def chunk_text(text, chunk_size=120, overlap=30):
    """
    Split a document into overlapping word-based chunks.

    Parameters
    ----------
    text : str
        Document text.

    chunk_size : int
        Maximum number of words per chunk.

    overlap : int
        Number of overlapping words between consecutive chunks.
    """

    words = str(text).split()

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunks.append(
            " ".join(words[start:end]) )

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks

In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# Semantic Chunk DataFrame
# Step 2: Split semantic documents while preserving metadata
# ──────────────────────────────────────────────────────────────────────────────
semantic_chunks = []

for _, row in dataset.iterrows():

    chunks = chunk_text(row["semantic_text"])

    for chunk_index, chunk in enumerate(chunks):

        semantic_chunks.append({

            # Unique chunk identifier
            "chunk_id":
                f"{row['QuestionID']}_chunk_{chunk_index}",

            # Parent document identifier
            "QuestionID":
                row["QuestionID"],

            # Metadata
            "Category":
                row["Category"],

            "QuestionType":
                row["QuestionType"],

            "QuestionTime":
                row["QuestionTime"],

            # Chunk order
            "chunk_index":
                chunk_index,

            # Chunk content
            "chunk_text":
                chunk,

            # Text used for embedding generation
            "search_text":
                (
                    f"Category: {row['Category']} "
                    f"Question Type: {row['QuestionType']} "
                    f"{chunk}"
                )

        }) 

semantic_chunks_df = pd.DataFrame(semantic_chunks)

print("Total Semantic Chunks:", len(semantic_chunks_df))

semantic_chunks_df.head()

Total Semantic Chunks: 113871


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C15Q2112_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,Question: - what are the dimensions of this it...,Category: Tools and Home Improvement Question ...
1,C9Q4595_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,Question: how much booze can it hold? Answer: ...,Category: Home and Kitchen Question Type: open...
2,C4Q7999_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,Question: will this case fit nokia lumia 520 A...,Category: Cell Phones and Accessories Question...
3,C8Q8916_chunk_0,C8Q8916,Health and Personal Care,open-ended,2014-04-25,0,"Question: when folded in the sitting position,...",Category: Health and Personal Care Question Ty...
4,C14Q905_chunk_0,C14Q905,Sports and Outdoors,open-ended,2015-04-15,0,Question: how long should i leave this on to g...,Category: Sports and Outdoors Question Type: o...


In [14]:
# ──────────────────────────────────────────────────────────────────────────────
# Lexical Chunk DataFrame
# Step 3: Split lexical documents while preserving metadata
# ──────────────────────────────────────────────────────────────────────────────
lexical_chunks = []

for _, row in dataset.iterrows():

    chunks = chunk_text(row["lexical_text"])

    for chunk_index, chunk in enumerate(chunks):

        lexical_chunks.append({

            # Unique chunk identifier
            "chunk_id":
                f"{row.name}_chunk_{chunk_index}",

            # Parent document identifier
            "QuestionID":
                row["QuestionID"],

            # Metadata
            "Category":
                row["Category"],

            "QuestionType":
                row["QuestionType"],

            "QuestionTime":
                row["QuestionTime"],

            # Chunk order
            "chunk_index":
                chunk_index,

            # Chunk content
            "chunk_text":
                chunk,

            # Text used for BM25 indexing
            "search_text":
                (
                    f"{row['Category']} "
                    f"{row['QuestionType']} "
                    f"{chunk}"
                )

        })

lexical_chunks_df = pd.DataFrame(lexical_chunks)

print("Total Lexical Chunks:", len(lexical_chunks_df))

lexical_chunks_df.head()

Total Lexical Chunks: 105096


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C15Q2112_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,dimensions item,Tools and Home Improvement open-ended dimensio...
1,C9Q4595_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,much booze hold poured booze measuring cup sun...,Home and Kitchen open-ended much booze hold po...
2,C4Q7999_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,case fit nokia lumia 520 yes fits great great ...,Cell Phones and Accessories open-ended case fi...
3,C8Q8916_chunk_0,C8Q8916,Health and Personal Care,open-ended,2014-04-25,0,folded sitting position high ground seat 30 in...,Health and Personal Care open-ended folded sit...
4,C14Q905_chunk_0,C14Q905,Sports and Outdoors,open-ended,2015-04-15,0,long leave get max sweat benefit keep thinking...,Sports and Outdoors open-ended long leave get ...


In [15]:
# =============================================================================
# Inspect generated chunks
# =============================================================================

print("Semantic Chunk Shape :", semantic_chunks_df.shape)
print("Lexical Chunk Shape  :", lexical_chunks_df.shape)

semantic_chunks_df.head(3)

Semantic Chunk Shape : (113871, 8)
Lexical Chunk Shape  : (105096, 8)


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C15Q2112_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,Question: - what are the dimensions of this it...,Category: Tools and Home Improvement Question ...
1,C9Q4595_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,Question: how much booze can it hold? Answer: ...,Category: Home and Kitchen Question Type: open...
2,C4Q7999_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,Question: will this case fit nokia lumia 520 A...,Category: Cell Phones and Accessories Question...


In [16]:
# =============================================================================
# Display one complete chunk
# =============================================================================

sample = 0

print("Chunk ID:")
print(semantic_chunks_df.loc[sample, "chunk_id"])

print("\nMetadata")
print("-------------------------")
print("QuestionID :", semantic_chunks_df.loc[sample, "QuestionID"])
print("Category   :", semantic_chunks_df.loc[sample, "Category"])
print("Type       :", semantic_chunks_df.loc[sample, "QuestionType"])
print("Time       :", semantic_chunks_df.loc[sample, "QuestionTime"])

print("\nChunk Text")
print("-------------------------")
print(semantic_chunks_df.loc[sample, "chunk_text"])

print("\nSearch Text")
print("-------------------------")
print(semantic_chunks_df.loc[sample, "search_text"])

Chunk ID:
C15Q2112_chunk_0

Metadata
-------------------------
QuestionID : C15Q2112
Category   : Tools and Home Improvement
Type       : open-ended
Time       : 2013-04-20

Chunk Text
-------------------------
Question: - what are the dimensions of this item? Answer:

Search Text
-------------------------
Category: Tools and Home Improvement Question Type: open-ended Question: - what are the dimensions of this item? Answer:


In [17]:
# =============================================================================
# Save the chunks dataset
# =============================================================================
semantic_chunks_df.to_csv( "semantic_chunks.csv", index=False)
lexical_chunks_df.to_csv("lexical_chunks.csv", index=False)

print("Semantic and Lexical chunks saved successfully.")

Semantic and Lexical chunks saved successfully.
